In [1]:
# --- Core Libraries ---
import torch
import pandas as pd
import numpy as np
import os
import json
import joblib
from pathlib import Path
import warnings
from tqdm.notebook import tqdm

# --- Transformers ---
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader

# --- NLP & ML Libraries from Phase 3 ---
import spacy
from textstat import flesch_reading_ease
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import Counter
from urllib.parse import urlparse
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# --- Google Gemini ---
import google.generativeai as genai

warnings.filterwarnings('ignore')
tqdm.pandas()

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
nlp = spacy.load("en_core_web_lg")

class AdvancedFeatureExtractor:
    """Extracts advanced NLP features for fake news detection."""
    def __init__(self):
        self.nlp = nlp
        self.sentiment_analyzer = SentimentIntensityAnalyzer()
        self.sensational_words = {
            'shocking', 'unbelievable', 'amazing', 'incredible', 'outrageous',
            'scandal', 'exposed', 'revealed', 'secret', 'hidden', 'truth',
            'must see', "you won't believe", 'exclusive', 'breaking', 'hate'
        }
        # This list of reliable sources isn't used in extract_all_features but is kept for completeness
        self.reliable_sources = {
            'bbc.com', 'reuters.com', 'apnews.com', 'npr.org',
            'thehindu.com', 'indianexpress.com', 'ndtv.com'
        }

    def extract_linguistic_features(self, text):
        doc = self.nlp(text)
        pos_counts = Counter(token.pos_ for token in doc)
        num_words = len(doc)
        if num_words == 0: num_words = 1

        features = {
            'text_length': len(text),
            'word_count': num_words,
            'sentence_count': len(list(doc.sents)),
            'avg_word_length': np.mean([len(token) for token in doc]) if num_words > 1 else 0,
            'flesch_reading_ease': flesch_reading_ease(text),
            'exclamation_count': text.count('!'),
            'question_count': text.count('?'),
            'caps_ratio': sum(1 for c in text if c.isupper()) / (len(text) + 1e-6),
            'noun_ratio': pos_counts.get('NOUN', 0) / num_words,
            'verb_ratio': pos_counts.get('VERB', 0) / num_words,
            'adj_ratio': pos_counts.get('ADJ', 0) / num_words,
            'sensational_word_ratio': sum(1 for word in doc if word.lower_ in self.sensational_words) / num_words,
        }
        return features

    def extract_sentiment_features(self, text):
        vader_scores = self.sentiment_analyzer.polarity_scores(text)
        return {
            'sentiment_compound': vader_scores['compound'],
            'sentiment_positive': vader_scores['pos'],
            'sentiment_negative': vader_scores['neg'],
        }

    def extract_entity_features(self, text):
        doc = self.nlp(text)
        entities = doc.ents
        return {
            'total_entities': len(entities),
            'person_count': sum(1 for ent in entities if ent.label_ == 'PERSON'),
            'org_count': sum(1 for ent in entities if ent.label_ == 'ORG'),
            'gpe_count': sum(1 for ent in entities if ent.label_ == 'GPE'), # Geopolitical Entity
        }

    def extract_all_features(self, text):
        # This method now combines all feature types, matching the original notebook
        all_features = {}
        all_features.update(self.extract_linguistic_features(text))
        all_features.update(self.extract_sentiment_features(text))
        all_features.update(self.extract_entity_features(text))
        return pd.Series(all_features)

In [4]:
class HybridEnsembleClassifier:
    """Combines DistilBERT with traditional ML models for enhanced prediction."""
    def __init__(self, distilbert_model, tokenizer, feature_scaler, rf_model, xgb_model, meta_learner, device='cpu'):
        self.distilbert_model = distilbert_model
        self.tokenizer = tokenizer
        self.feature_scaler = feature_scaler
        self.device = device
        self.rf_model = rf_model
        self.xgb_model = xgb_model
        self.meta_learner = meta_learner

    def get_distilbert_predictions(self, texts):
        all_probs = []
        # Ensure texts is a list of strings
        if not isinstance(texts, list): texts = [texts]
        loader = DataLoader([{"text": t} for t in texts], batch_size=16)

        with torch.no_grad():
            for batch in loader:
                inputs = self.tokenizer(
                    batch["text"], truncation=True, padding=True, max_length=512, return_tensors='pt'
                ).to(self.device)
                outputs = self.distilbert_model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                all_probs.extend(probs.cpu().numpy())
        return np.array(all_probs)

    def predict_proba(self, texts, advanced_features):
        bert_probs = self.get_distilbert_predictions(texts)
        rf_probs = self.rf_model.predict_proba(advanced_features)
        xgb_probs = self.xgb_model.predict_proba(advanced_features)

        meta_features = np.column_stack([bert_probs, rf_probs, xgb_probs])
        final_probs = self.meta_learner.predict_proba(meta_features)
        return final_probs

print("Setup complete. Custom classes from Phase 3 are now defined.")

Setup complete. Custom classes from Phase 3 are now defined.


In [5]:
# --- Define Paths to Artifacts ---
DISTILBERT_PATH = Path("models/final_fake_news_detector")
ENSEMBLE_PATH = Path("models/ensemble_components")
SCALER_PATH = Path("models/feature_scaler.pkl")

# --- Load Phase 2: Fine-Tuned DistilBERT Model ---
print(f"Loading DistilBERT model from: {DISTILBERT_PATH}")
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(DISTILBERT_PATH)
distilbert_model = DistilBertForSequenceClassification.from_pretrained(DISTILBERT_PATH)
distilbert_model.to(device)
distilbert_model.eval()
print("✅ DistilBERT model and tokenizer loaded.")

# --- Load Phase 3: Hybrid Ensemble Components ---
print(f"\nLoading Hybrid Ensemble components from: {ENSEMBLE_PATH}")
feature_scaler = joblib.load(SCALER_PATH)
rf_model = joblib.load(ENSEMBLE_PATH / 'rf_model.pkl')
xgb_model = joblib.load(ENSEMBLE_PATH / 'xgb_model.pkl')
meta_learner = joblib.load(ENSEMBLE_PATH / 'meta_learner.pkl')
print("✅ Feature scaler and ensemble models (RF, XGB, Meta-learner) loaded.")

# --- Initialize Feature Extractor and Hybrid Model ---
feature_extractor = AdvancedFeatureExtractor()

hybrid_model = HybridEnsembleClassifier(
    distilbert_model=distilbert_model,
    tokenizer=distilbert_tokenizer,
    feature_scaler=feature_scaler,
    rf_model=rf_model,
    xgb_model=xgb_model,
    meta_learner=meta_learner,
    device=device
)
print("\n✅ All models and artifacts are loaded and ready for the pipeline.")

Loading DistilBERT model from: models/final_fake_news_detector
✅ DistilBERT model and tokenizer loaded.

Loading Hybrid Ensemble components from: models/ensemble_components
✅ Feature scaler and ensemble models (RF, XGB, Meta-learner) loaded.

✅ All models and artifacts are loaded and ready for the pipeline.


In [6]:
class UnifiedAnalysisPipeline:
    """
    A pipeline that runs an article through all trained models to generate a
    comprehensive analysis context.
    """
    def __init__(self, hybrid_model, feature_extractor):
        self.hybrid_model = hybrid_model
        self.feature_extractor = feature_extractor
        self.labels = ['Reliable', 'Unreliable']

    def analyze(self, article_text: str) -> dict:
        """
        Processes a single news article and returns a dictionary of analytical signals.
        """
        print(f"Analyzing text: '{article_text[:80]}...'")

        # 1. Extract Advanced Features (Sentiment, NER, Linguistics)
        # We need a DataFrame-like structure for the extractor
        temp_df = pd.DataFrame({'text': [article_text]})
        adv_features_series = self.feature_extractor.extract_all_features(article_text)
        adv_features_df = adv_features_series.to_frame().T

        # 2. Scale the features
        scaled_adv_features = self.hybrid_model.feature_scaler.transform(adv_features_df)

        # 3. Get Hybrid Model Prediction
        hybrid_probs = self.hybrid_model.predict_proba([article_text], scaled_adv_features)[0]
        hybrid_pred_idx = np.argmax(hybrid_probs)
        hybrid_confidence = hybrid_probs[hybrid_pred_idx]
        hybrid_label = self.labels[hybrid_pred_idx]

        # 4. Compile all signals into a structured dictionary
        analysis_context = {
            "prediction_label": hybrid_label,
            "prediction_confidence": float(f"{hybrid_confidence:.4f}"),
            "probabilities": {
                "reliable": float(f"{hybrid_probs[0]:.4f}"),
                "unreliable": float(f"{hybrid_probs[1]:.4f}")
            },
            "linguistic_signals": {
                "sentiment_score": float(f"{adv_features_df['sentiment_compound'].iloc[0]:.3f}"),
                "sensationalism_ratio": float(f"{adv_features_df['sensational_word_ratio'].iloc[0]:.3f}"),
                "readability_score": float(f"{adv_features_df['flesch_reading_ease'].iloc[0]:.1f}"),
                "capitalization_ratio": float(f"{adv_features_df['caps_ratio'].iloc[0]:.3f}")
            },
            "entity_signals": {
                "total_entities": int(adv_features_df['total_entities'].iloc[0]),
                "people_mentioned": int(adv_features_df['person_count'].iloc[0]),
                "organizations_mentioned": int(adv_features_df['org_count'].iloc[0])
            }
        }

        return analysis_context

# --- Initialize the pipeline ---
unified_pipeline = UnifiedAnalysisPipeline(hybrid_model, feature_extractor)
print("\n✅ Unified Analysis Pipeline initialized.")


✅ Unified Analysis Pipeline initialized.


In [7]:
!pip install -q -U google-generativeai

# --- Configure Gemini API ---
try:
    # Use the same key from your Phase 1 notebook
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "xxx")
    if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY":
        print("🛑 WARNING: Please set your Gemini API key.")
    else:
        genai.configure(api_key=GEMINI_API_KEY)
        generative_model = genai.GenerativeModel('gemini-2.5-flash')
        print("✅ Gemini API configured successfully with 'gemini-2.5-flash'.")
except Exception as e:
    print(f"🔥 Error configuring Gemini API: {e}")
    generative_model = None


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
✅ Gemini API configured successfully with 'gemini-2.5-flash'.


In [8]:
def generate_explanation(analysis_context: dict, article_text: str) -> str:
    """
    Uses the Gemini model to generate a plain-language explanation based on the
    analysis signals from our ML pipeline.
    """
    if not generative_model:
        return "Generative model is not available. Please check API key and configuration."

    # Convert the context dictionary to a clean string for the prompt
    context_str = json.dumps(analysis_context, indent=2)

    # This prompt is engineered to guide the LLM's response
    prompt = f"""
    You are an AI news analysis assistant. Your task is to provide a clear, neutral, and educational explanation for why a news article has been flagged as potentially unreliable. Do not make a definitive judgment; instead, highlight the signals that your underlying models detected.

    Here is the analysis data from my machine learning models:
    ```json
    {context_str}
    ```

    Here is a snippet of the article text:
    "{article_text[:500]}..."

    Based on the data above, generate a brief, easy-to-understand summary for the end-user. Structure your response with the following sections:
    1.  **Overall Assessment:** Start with a summary sentence stating the model's confidence level (e.g., "Our analysis suggests this article has a [high/moderate] probability of being unreliable.").
    2.  **Key Signals Detected:** Create a bulleted list explaining the specific signals that contributed to this assessment. Translate the data points into simple concepts. For example:
        - If sentiment_score is very low (e.g., < -0.5), mention "The article uses strongly negative and emotionally charged language."
        - If sensationalism_ratio is high (e.g., > 0.05), mention "It contains words often found in sensational or 'clickbait' headlines."
        - If capitalization_ratio is high (e.g., > 0.1), mention "The text uses excessive capitalization, which can be a sign of low-quality or biased reporting."
        - Mention the number of people or organizations if it's relevant.
    3.  **Recommendation:** Conclude with a neutral recommendation, such as "We recommend cross-referencing this information with other established news sources."

    Do not include any text before "Overall Assessment". Keep the tone helpful and educational.
    """

    try:
        response = generative_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"🔥 An error occurred while generating the explanation: {e}"

print("✅ Explanation generation function is ready.")

✅ Explanation generation function is ready.


In [9]:
# --- Sample Articles for Demonstration ---
sample_articles = [
    {
        "title": "Clearly Unreliable Article",
        "text": "SHOCKING REVELATION!! Doctors are STUNNED by this one weird trick to lose weight. The government is trying to HIDE this secret from you! You won't believe what they found. This is the truth they don't want you to know!!"
    },
    {
        "title": "Clearly Reliable Article",
        "text": "The Indian Space Research Organisation (ISRO) has announced that the upcoming Chandrayaan-4 mission will focus on lunar sample return. The mission, scheduled for late 2028, aims to collect regolith from the Moon's south pole to study water ice deposits, according to a statement released by the agency's chairman."
    },
    {
        "title": "Subtle Unreliable Article (Political Bias)",
        "text": "The opposition party's latest proposal is nothing short of a complete disaster for the nation's economy. Their reckless spending plan will undoubtedly lead to runaway inflation and destroy jobs, proving once again they are unfit to govern. Every citizen should be outraged by this irresponsible scheme."
    },
    {
        "title": "Standard News Report",
        "text": "The Reserve Bank of India's Monetary Policy Committee today decided to keep the repo rate unchanged at 6.5 percent, citing persistent inflationary pressures. The committee noted that while economic growth remains robust, monitoring food price inflation remains a key priority for the coming quarter."
    }
]

# --- Run the Full Pipeline for each sample ---
for article in sample_articles:
    print("="*80)
    print(f"📰 PROCESSING: {article['title']}")
    print("="*80)

    # Step 1: Get structured data from our ML pipeline
    analysis_results = unified_pipeline.analyze(article['text'])
    print("\n--- 📊 Raw ML Pipeline Output ---")
    print(json.dumps(analysis_results, indent=2))

    # Step 2: Get the human-readable explanation from Gemini
    print("\n--- 🤖 Gemini-Generated Explanation ---")
    explanation = generate_explanation(analysis_results, article['text'])
    print(explanation)
    print("\n\n")

📰 PROCESSING: Clearly Unreliable Article
Analyzing text: 'SHOCKING REVELATION!! Doctors are STUNNED by this one weird trick to lose weight...'

--- 📊 Raw ML Pipeline Output ---
{
  "prediction_label": "Unreliable",
  "prediction_confidence": 0.561,
  "probabilities": {
    "reliable": 0.439,
    "unreliable": 0.561
  },
  "linguistic_signals": {
    "sentiment_score": -0.893,
    "sensationalism_ratio": 0.062,
    "readability_score": 90.6,
    "capitalization_ratio": 0.151
  },
  "entity_signals": {
    "total_entities": 0,
    "people_mentioned": 0,
    "organizations_mentioned": 0
  }
}

--- 🤖 Gemini-Generated Explanation ---
Overall Assessment: Our analysis suggests this article has a moderate probability of being unreliable.

Key Signals Detected:
*   The article uses strongly negative and emotionally charged language.
*   It contains words often found in sensational or 'clickbait' headlines.
*   The text uses excessive capitalization, which can be a sign of low-quality or biased 